In [1]:
from vllm import LLM, EngineArgs
from vllm.utils import FlexibleArgumentParser
from vllm.sampling_params import SamplingParams
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["VLLM_PP_LAYER_PARTITION"] = "4,4,4,4,4,4,4,2"
import torch

INFO 09-08 06:29:44 [__init__.py:244] Automatically detected platform cuda.


In [2]:
# 提取采样参数
max_tokens = 512
temperature = 0.7
top_p = 0.9
top_k = 50

In [3]:
# 构建采样对象
sampling_params = SamplingParams(
    max_tokens=max_tokens,
    temperature=temperature,
    top_p=top_p,
    top_k=top_k,
)

In [4]:
def create_parser():
    parser = FlexibleArgumentParser()
    EngineArgs.add_cli_args(parser)

    # 默认模型路径改成本地 Qwen2.5-7B-Instruct
    parser.set_defaults(model="/workspace/zimoliu/models/Qwen2.5-7B-Instruct")

    # 采样参数（命令行可覆盖）
    sampling_group = parser.add_argument_group("Sampling parameters")
    sampling_group.add_argument("--max-tokens", type=int, default=512)
    sampling_group.add_argument("--temperature", type=float, default=0.7)
    sampling_group.add_argument("--top-p", type=float, default=0.9)
    sampling_group.add_argument("--top-k", type=int, default=50)

    return parser

In [5]:
def main(args: dict):
    # 提取采样参数
    max_tokens = args.pop("max_tokens")
    temperature = args.pop("temperature")
    top_p = args.pop("top_p")
    top_k = args.pop("top_k")

    # 构建采样对象
    sampling_params = SamplingParams(
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
    )

    # 创建 LLM 实例
    llm = LLM(**args)

    print(">>> Qwen2.5-7B-Instruct 已加载，输入 q 退出对话 <<<")

    while True:
        try:
            user_input = input("\n你：").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n再见！")
            break

        if user_input.lower() == "q":
            print("再见！")
            break
        if not user_input:
            continue

        # 构造单轮对话（无历史）
        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_input},
        ]

        # 调用 vLLM 的 chat 接口
        outputs = llm.chat([conversation], sampling_params, use_tqdm=False)

        # 取第一条结果
        assistant_reply = outputs[0].outputs[0].text.strip()
        print("\n助手：", assistant_reply)

In [ ]:
# if __name__ == "__main__":
parser = create_parser()
args = vars(parser.parse_args())
main(args)

In [2]:
# Notebook 单 cell 版：Qwen2.5-7B-Instruct 对话
from vllm import LLM
from vllm.sampling_params import SamplingParams



In [2]:
# ========== 1. 默认参数 ==========
# MODEL_PATH = "/workspace/zimoliu/models/Qwen2.5-7B-Instruct"
# SAMPLING_KWARGS = dict(
#     max_tokens=2048,
#     temperature=0.7,
#     top_p=0.9,
#     top_k=50,
# )

MODEL_PATH = "/sharedata/zimoliu/models/Jamba-v0.1"
SAMPLING_KWARGS = dict(
    max_tokens=128,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
)


In [2]:
MODEL_PATH = "/sharedata/zimoliu/ckpts/jamba_60B_128k_v6_1node_pp8_ep1_official_ckpt38000/hf"
SAMPLING_KWARGS = dict(
    max_tokens=128,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
)

In [ ]:
# ========== 2. 加载模型 ==========
print(">>> 正在加载模型，请稍候...")
llm = LLM(
    model=MODEL_PATH,
    pipeline_parallel_size=8,
    # 如有其它 EngineArgs，可在此追加，例如：
    # tensor_parallel_size=1,
    # gpu_memory_utilization=0.8,
    # 关键：关闭 flashinfer，用原生 top-k/top-p
    # enforce_eager=True,
    # 或者
    # disable_flashinfer=True,
)
sampling_params = SamplingParams(**SAMPLING_KWARGS)
print(">>> 模型已就绪，输入 q 退出对话 <<<")



>>> 正在加载模型，请稍候...
-------- 即将传给 ModelConfig 的参数 --------
{'model': '/sharedata/zimoliu/ckpts/jamba_60B_128k_v6_1node_pp8_ep1_official_ckpt38000/hf',
 'hf_config_path': None,
 'task': 'auto',
 'tokenizer': None,
 'tokenizer_mode': 'auto',
 'trust_remote_code': False,
 'allowed_local_media_path': '',
 'dtype': 'auto',
 'seed': None,
 'revision': None,
 'code_revision': None,
 'rope_scaling': {},
 'rope_theta': None,
 'hf_token': None,
 'hf_overrides': {},
 'tokenizer_revision': None,
 'max_model_len': None,
 'quantization': None,
 'enforce_eager': False,
 'max_seq_len_to_capture': 8192,
 'max_logprobs': 20,
 'disable_sliding_window': False,
 'disable_cascade_attn': False,
 'skip_tokenizer_init': False,
 'enable_prompt_embeds': False,
 'served_model_name': None,
 'limit_mm_per_prompt': {},
 'media_io_kwargs': {},
 'use_async_output_proc': True,
 'config_format': 'auto',
 'mm_processor_kwargs': None,
 'disable_mm_preprocessor_cache': False,
 'override_neuron_config': {},
 'override_pooler_

You are using a model of type jambadoe to instantiate a model of type jamba_doe. This is not supported for all configurations of models and can yield errors.


### get_config config_dict: {'architectures': ['JambaDoEForCausalLM'], 'attention_dropout': 0.0, 'bos_token_id': 151643, 'eos_token_id': 151645, 'hidden_act': 'silu', 'hidden_size': 4096, 'initializer_range': 0.02, 'intermediate_size': 8192, 'max_position_embeddings': 131072, 'max_window_layers': 30, 'model_type': 'jambadoe', 'num_attention_heads': 32, 'num_hidden_layers': 30, 'num_key_value_heads': 32, 'rms_norm_eps': 1e-05, 'rope_theta': 5000000.0, 'sliding_window': 131072, 'tie_word_embeddings': False, 'torch_dtype': 'bfloat16', 'transformers_version': '4.43.1', 'use_cache': True, 'use_sliding_window': False, 'vocab_size': 151936, '_commit_hash': None}
### hf_config: JambaDoEConfig {
  "architectures": [
    "JambaDoEForCausalLM"
  ],
  "attention_dropout": 0.0,
  "attn_layer_offset": 3,
  "attn_layer_period": 4,
  "bos_token_id": 151643,
  "conv_attention_kernel_size": 4,
  "eos_token_id": 151645,
  "expert_layer_offset": 0,
  "expert_layer_period": 1,
  "hidden_act": "silu",
  "hi

INFO 09-08 06:30:02 [config.py:848] This model supports multiple tasks: {'classify', 'generate', 'embed', 'reward'}. Defaulting to 'generate'.
### model_info: _ModelInfo(architecture='JambaDoEForCausalLM', is_text_generation_model=True, is_pooling_model=True, supports_cross_encoding=False, supports_multimodal=False, supports_pp=True, has_inner_state=True, is_attention_free=False, is_hybrid=True, has_noops=False, supports_transcription=False, supports_v0_only=True) arch: JambaDoEForCausalLM
INFO 09-08 06:30:02 [config.py:1479] Using max model len 131072
WARNING 09-08 06:30:02 [arg_utils.py:1782] ['JambaDoEForCausalLM'] is not supported by the V1 Engine. Falling back to V0. 
WARNING 09-08 06:30:02 [arg_utils.py:1578] Chunked prefill is enabled by default for models with max_model_len > 32K. Chunked prefill might not work with some features or models. If you encounter any issues, please disable by launching with --enable-chunked-prefill=False.
INFO 09-08 06:30:02 [config.py:2292] Chunked 

Loading safetensors checkpoint shards:   0% Completed | 0/30 [00:00<?, ?it/s]


(VllmWorkerProcess pid=581125) (VllmWorkerProcess pid=581122) (VllmWorkerProcess pid=581112) ### rank: 0, jamba doe weight loader weights all name: decoder.layers.0.mixer.D, shape: torch.Size([128])

### rank: 0, jamba doe weight loader weights all name: decoder.layers.0.mixer.A_log, shape: torch.Size([128])

### rank: 0, jamba doe weight loader weights all name: decoder.layers.0.norm.weight, shape: torch.Size([4096])

### rank: 0, jamba doe weight loader weights all name: decoder.layers.0.mixer.dt_bias, shape: torch.Size([128])

### rank: 0, jamba doe weight loader weights all name: decoder.layers.0.mixer.conv1d.bias, shape: torch.Size([10240])

### rank: 0, jamba doe weight loader weights all name: decoder.layers.0.mixer.norm.weight, shape: torch.Size([8192])

### rank: 0, jamba doe weight loader weights all name: decoder.layers.0.mlp.router.weight, shape: torch.Size([16, 4096])

### rank: 0, jamba doe weight loader weights all name: decoder.layers.0.mixer.conv1d.weight, shape: torch

In [ ]:
print(123)

In [4]:
def chat_loop():
    while True:
        try:
            user_input = input("").strip()          # 去掉提示符
        except (KeyboardInterrupt, EOFError):
            print("\n再见！")
            break
        if user_input.lower() == "q":
            print("再见！")
            break
        if not user_input:
            continue

        # 打印用户输入
        print(f"\n你：{user_input}")

        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_input},
        ]
        outputs = llm.generate(prompts=user_input, sampling_params=sampling_params)
        # outputs = llm.chat([conversation], sampling_params, use_tqdm=False)
        assistant_reply = outputs[0].outputs[0].text.strip()
        print(f"助手：{assistant_reply}")

In [5]:
# 运行对话
chat_loop()
del llm
torch.cuda.empty_cache()
torch.cuda.synchronize()


你：你好，你是什么模型


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.14s/it, est. speed input: 8.82 toks/s, output: 54.68 toks/s]


助手：？

你：什么是机器学习？答：


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.48s/it, est. speed input: 7.44 toks/s, output: 86.58 toks/s]


助手：机器学习是一门使用计算机算法从数据中学习，通常是通过一些统计方法。

什么是监督学习？答：监督学习是一种机器学习，其中机器学习模型从有标签的数据中学习。

什么是无监督学习？答：无监督学习是一种机器学习，其中机器学习模型从无标签的数据中学习。

什么是

你：一个一周训练6天的健身计划：


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.48s/it, est. speed input: 10.84 toks/s, output: 86.74 toks/s]


助手：*   周一：肩胛骨
*   周二：腹部
*   周三：肩胛骨
*   周四：腹部
*   周五：肩胛骨
*   周六：腹部


我的一周健身计划：


*   周一：肩胛骨
*   周二：腹部
*   周三：肩
再见！


In [5]:
# 运行对话
chat_loop()
del llm
torch.cuda.empty_cache()
torch.cuda.synchronize()


你：你好，你是什么模型
INFO 08-14 04:42:51 [chat_utils.py:444] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


ValueError: As of transformers v4.44, default chat template is no longer allowed, so you must provide a chat template if the tokenizer does not define one.

In [8]:
del llm
torch.cuda.empty_cache()
torch.cuda.synchronize()

NameError: name 'llm' is not defined